# Train + Validate the Full Combined Agent on Kaggle GPU

Clones `approach/full-combo` and runs everything there. Four signals in
one agent:

1. **Candidate-filtering** -- dictionary words still consistent with the
   board + wrong guesses, sharp once the pool narrows.
2. **Character n-gram fallback** -- CandidateAgent's own forward/backward
   n-gram model, reused directly (no dictionary match needed).
3. **BiLSTM** (attention + guessed-wrong/remaining features + cosine LR
   decay) -- the strongest neural variant built so far.
4. **Vowel-ratio guard** -- once over half the *revealed* letters are
   vowels, stop guessing further vowels.

All three numeric signals (candidate/ngram/neural) are normalized to
proper distributions before blending -- the plain ensemble branch had a
real bug where an unnormalized neural score numerically dominated
regardless of its intended blend weight; fixed here from the start.

**Before running:** in the notebook's Settings panel (right sidebar), set
**Accelerator = GPU T4 x2** (or any GPU) and **Internet = On** (needed to `git clone`).

In [ ]:
import torch
print('torch', torch.__version__, 'cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device:', torch.cuda.get_device_name(0))
else:
    print('WARNING: no GPU detected -- check Settings > Accelerator in the sidebar')

In [ ]:
REPO_URL = "https://github.com/Sahoo-Achyutananda/MELTWATER---HACKATHON.git"
BRANCH = "approach/full-combo"

!rm -rf repo
!git clone --branch $BRANCH --single-branch $REPO_URL repo
%cd repo/brand-buzzword-hackathon
!ls

## Use the official competition dataset

The cloned repo carries its own copy of train.txt/test.txt (downloaded from
this same competition earlier), but overwrite them here so this notebook
verifiably sources data straight from Kaggle's own `/kaggle/input/`, not an
external GitHub copy -- same content, no ambiguity for anyone reviewing it.

In [ ]:
import shutil
shutil.copy("/kaggle/input/competitions/brand-buzzword-hackathon/train.txt", "train.txt")
shutil.copy("/kaggle/input/competitions/brand-buzzword-hackathon/test.txt", "test.txt")
print("train.txt and test.txt overwritten with the official competition dataset from /kaggle/input/")

## Train the BiLSTM half

Masked-language-model objective + synthetic guessed-wrong/remaining
features + cosine LR decay. Bump `--epochs` up if you want -- the
schedule automatically stretches to match whatever you pass.

In [ ]:
!python src/train_bilstm.py --epochs 20

## Validate the combined agent

Same held-out-train.txt methodology as every other branch. Compare
against candidate-filtering alone (39-40%), BiLSTM alone (47.7%), and the
plain ensemble (candidate+BiLSTM, post-fix) -- this needs to beat all
three to justify the added complexity.

In [ ]:
!python src/validate_combined.py

## Generate submission.csv

Plays the actual game against every word in test.txt using the combined
agent still in this session. Sandbox leaderboard checkpoint only -- per
the competition's Final Judgement policy, final hiring decisions re-run
the submitted model/notebook against a separate private word list.

250,000 words, one game at a time -- prints progress every 20,000 words
with an ETA. Expect this to run slower than any single-signal branch,
since every turn does candidate matching + n-gram scoring + a neural
forward pass.

In [ ]:
!python src/generate_submission_combined.py

## Save outputs

Anything under `/kaggle/working/` is downloadable from the notebook's
Output tab after the run finishes.

In [ ]:
import shutil
shutil.copy("src/bilstm_attn_feat_masker.pt", "/kaggle/working/bilstm_attn_feat_masker.pt")
shutil.copy("submission.csv", "/kaggle/working/submission.csv")
print("saved bilstm_attn_feat_masker.pt and submission.csv to /kaggle/working/ -- download from the Output tab")